# Has COVID-19 impacted reviews of scented candles negatively?

A famous bit of pandemic data sleuthing: scented-candle Amazon ratings dipped
in 2020 while unscented ones held steady — and reviews complaining the candle
has *no scent* climbed as anosmia spread. Ported from the BeakerX-era
`Candles.ipynb` in
[groovy-data-science](https://github.com/paulk-asert/groovy-data-science).

The review datasets (`Scented_all.xlsx`, `Unscented_all.xlsx`) sit alongside
this notebook and are read with [Apache POI](https://poi.apache.org/) via
`@Grab` — a dozen lines replace the BeakerX-era tablesaw-excel dependency.
Charts are hand-rolled SVG: monthly average line, daily average dots, and a
dashed marker where COVID-19 was first reported (20 Jan 2020).

In [ ]:
@Grab('org.apache.poi:poi-ooxml:5.4.1')
import org.apache.poi.ss.usermodel.WorkbookFactory
readXlsx = { File file ->
    WorkbookFactory.create(file).withCloseable { wb ->
        def sheet = wb.getSheetAt(0)
        (1..sheet.lastRowNum).collect { ri ->
            def row = sheet.getRow(ri)
            [id    : row.getCell(0).numericCellValue as int,
             date  : row.getCell(1).localDateTimeCellValue.toLocalDate(),
             rating: row.getCell(2).numericCellValue,
             review: row.getCell(3)?.toString() ?: '']
        }
    }
}
'deps ready'

In [ ]:
scented = readXlsx(new File('Scented_all.xlsx'))
unscented = readXlsx(new File('Unscented_all.xlsx'))
"scented: ${scented.size()} reviews, unscented: ${unscented.size()} reviews"

In [ ]:
scented.take(3).collect { it.subMap(['id', 'date', 'rating']) + [review: it.review.take(60) + '...'] }

A reusable chart closure: daily average ratings as faint dots, monthly
averages as a thick line, restricted to the top-3 candles and 2017 onward,
with the COVID-19 marker:

In [ ]:
import java.time.LocalDate
covidReported = LocalDate.of(2020, 1, 20)
ratingChart = { List rows, String title, String color ->
    def from2017 = rows.findAll { it.date >= LocalDate.of(2017, 1, 1) && it.id <= 3 }
    def daily = from2017.groupBy { it.date }.collectEntries { d, rs -> [d, rs*.rating.average()] }
    def monthly = from2017.groupBy { LocalDate.of(it.date.year, it.date.month, 15) }
                          .collectEntries { d, rs -> [d, rs*.rating.average()] }.sort { it.key }
    long x0 = LocalDate.of(2017, 1, 1).toEpochDay()
    long x1 = LocalDate.of(2020, 12, 1).toEpochDay()
    def sx = { LocalDate d -> (40 + 560 * (d.toEpochDay() - x0) / (x1 - x0)).round(1) }
    def sy = { double v -> (330 - 300 * (v - 1) / 4).round(1) }
    def dots = daily.collect { d, v -> "<circle cx='${sx(d)}' cy='${sy(v as double)}' r='2' fill='$color' fill-opacity='0.25'/>" }.join('\n')
    def path = monthly.collect { d, v -> "${sx(d)},${sy(v as double)}" }.join(' ')
    def years = (2017..2020).collect { y ->
        def xx = sx(LocalDate.of(y, 1, 1))
        "<line x1='$xx' y1='330' x2='$xx' y2='335' stroke='gray'/><text x='$xx' y='348' font-size='11' fill='gray'>$y</text>"
    }.join('\n')
    def yticks = (1..5).collect { v -> "<text x='18' y='${sy(v as double) + 4}' font-size='11' fill='gray'>$v</text>" }.join('\n')
    def cx = sx(covidReported)
    """<svg xmlns='http://www.w3.org/2000/svg' width='640' height='360'>
<text x='40' y='16' font-size='13' fill='gray'>$title</text>
$dots
<polyline points='$path' fill='none' stroke='$color' stroke-width='3'/>
<line x1='$cx' y1='25' x2='$cx' y2='330' stroke='#e41a1c' stroke-width='2' stroke-dasharray='6,4'/>
<text x='${cx + 4}' y='40' font-size='11' fill='#e41a1c'>COVID-19 reported</text>
$years
$yticks
</svg>"""
}
'chart helper ready'

In [ ]:
displaySvg(ratingChart(scented, 'Top 3 scented candles - average Amazon rating (1-5)', '#377eb8'))
null

The same chart for unscented candles shows no comparable post-COVID dip:

In [ ]:
displaySvg(ratingChart(unscented, 'Top 3 unscented candles - average Amazon rating (1-5)', '#984ea3'))
null

## "It has no scent!"

If losing your sense of smell drives the dip, complaints should say so.
Flag 2020 reviews of the top-5 scented candles matching *no scent / no smell /
faint scent*-style phrases, month by month:

In [ ]:
candidates = ['[Nn]o scent', '[Nn]o smell', '[Dd]oes not smell like', "[Dd]oesn't smell like", "[Cc]an't smell",
              '[Cc]annot smell', '[Ff]aint smell', '[Ff]aint scent', "[Dd]on't smell", '[Ll]ike nothing']
monthStats = scented.findAll { it.date.year == 2020 }
        .groupBy { it.date.month }.sort { it.key.value }.collect { month, rs ->
    def matches = rs.count { r -> candidates.any { r.review =~ it } }
    double prop = matches / rs.size()
    [month: "${month.value}-${month.toString().toLowerCase().capitalize()}", reviews: rs.size(),
     noScent: matches, proportion: prop.round(3),
     stdErr: Math.sqrt(prop * (1 - prop) / rs.size()).round(3)]
}
monthStats

In [ ]:
top = monthStats.collect { it.proportion + it.stdErr }.max() * 1.15
sy = { v -> (320 - 290 * v / top).round(1) }
barW = (560 / monthStats.size()).round(1)
bars = monthStats.withIndex().collect { m, i ->
    def x = (40 + i * barW).round(1)
    def mid = (x + (barW - 6) / 2).round(1)
    "<rect x='$x' y='${sy(m.proportion)}' width='${(barW - 6).round(1)}' height='${(320 - sy(m.proportion)).round(1)}' fill='#377eb8'><title>${m.month}: ${m.noScent}/${m.reviews} = ${m.proportion}</title></rect>" +
    "<line x1='$mid' y1='${sy(m.proportion - m.stdErr)}' x2='$mid' y2='${sy(m.proportion + m.stdErr)}' stroke='#333' stroke-width='2'/>" +
    "<text x='$mid' y='334' font-size='11' fill='gray' text-anchor='middle'>${m.month.split('-')[1][0]}</text>"
}.join('\n')
displaySvg("<svg xmlns='http://www.w3.org/2000/svg' width='640' height='360'>\n" +
    "<text x='40' y='16' font-size='13' fill='gray'>Proportion of top-5 scented candle reviews mentioning lack of scent, 2020 (whiskers = ±1 SE)</text>\n" +
    "$bars\n</svg>")
null